## Description

This notebook uses heuristic and learned perceptual evaluation metrics to quantify the quality of synthetic image datasets for deepfake detector training.

We are most interested in measuring
1. realism (how indistinguishable a synthetic image is from a genuine image)
2. intra-dataset image diversity (degree of variation in image subject, style, color, texture, complexity, etc.) 

## Set-up environment

First, follow the set up instructions in the subnet repo Readme (make sure you have finished running 'python download_data.py')

We are also using Learned Perceptual Image Patch Similarity (LPIPS) metric, which can be installed with:

pip install lpips

https://github.com/richzhang/PerceptualSimilarity

In [1]:
%%time
# Standard library imports
import datetime
import json
import logging
import multiprocessing
import os
import random
import time
from multiprocessing import Pool, cpu_count
from pathlib import Path
from typing import List, Dict, Tuple, Any

# Related third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from PIL import Image, ImageFilter
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModel, pipeline
from torch.nn.functional import cosine_similarity
import lpips

# Project-specific or specialized libraries
from bitmind.constants import DATASET_META
from bitmind.image_dataset import ImageDataset
from utils import image_utils

/root/miniconda3/envs/bitmind/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-07-15 22:11:04.866294: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-15 22:11:04.888735: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-15 22:11:04.888811: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-15 22:11:04.908597: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow bin

CPU times: user 6.85 s, sys: 1.92 s, total: 8.77 s
Wall time: 8.5 s


### Configs


In [2]:
%%time
torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"

CPU times: user 24.6 ms, sys: 3.09 ms, total: 27.7 ms
Wall time: 190 ms


### Load Real Image Datasets

In [3]:
%%time
print("Loading real datasets")
real_image_datasets = [
    ImageDataset(ds['path'], 'test', ds.get('name', None), ds['create_splits'])
    for ds in DATASET_META['real']
]
real_image_datasets

Loading real datasets
CPU times: user 4.42 s, sys: 13.8 s, total: 18.2 s
Wall time: 26.2 s


### Create Directory of Real Images

In [4]:
def save_image(args):
    index, image, directory_path = args  # Unpack the tuple
    filename = f"{directory_path}/{index}.jpg"
    if image is not None and not os.path.exists(filename):
        image.save(filename)
        return f"Saved {filename}"
    return f"{filename} already exists or no image available"

def sample_images(directory_path : str, index_range, dataset : ImageDataset, n : int):
    print("Process ID: " + str(os.getpid()))
    num_samples = 0
    for idx in index_range:
        try:
            item = dataset[idx]
            if item['image'] is not None:
                save_image((idx, item['image'], directory_path))
                num_samples += 1
        except IndexError:
            # We reached the end of the dataset
            break
        if num_samples == n: return

def get_sample_list(directory_path : str, dataset: ImageDataset, n: int):
    num_processes = max(1, cpu_count() - 1)
    print('Num processes:', num_processes)
    dataset_length = len(dataset)
    
    # Calculate the number of items each process should handle
    items_per_process = dataset_length // num_processes
    # Create the ranges for each process
    ranges = [(i * items_per_process, min((i + 1) * items_per_process, dataset_length)) for i in range(num_processes)]
    # Initialize a multiprocessing pool
    with multiprocessing.Pool(processes=num_processes) as pool:
        # Each process will get a range of indices to process
        pool.starmap(sample_images, [(directory_path, range(start, end), dataset, n // num_processes) for start, end in ranges])

def create_eval_image_dataset(directory_path, dataset, n_samples):
    # Ensure the directory exists
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
        
    get_sample_list(directory_path, dataset, n_samples)

In [11]:
%%time
directory_path = os.getcwd()+'/test_data/eval_large_real_image_dataset_test/'
create_eval_image_dataset(directory_path, real_image_datasets[0], 1000)

Num processes: 15
Process ID: 14841Process ID: 14840Process ID: 14843


Process ID: 14844Process ID: 14845Process ID: 14846


Process ID: 14847Process ID: 14848

Process ID: 14849Process ID: 14851Process ID: 14850


Process ID: 14852
Process ID: 14853Process ID: 14854
Process ID: 14855

CPU times: user 45.1 ms, sys: 240 ms, total: 285 ms
Wall time: 37.8 s


# Vision Transformer Embedding + Cosine Similarity Metric

https://huggingface.co/docs/transformers/en/tasks/image_feature_extraction

In [3]:
dataset_directory_path = os.getcwd()+'/test_data/eval_large_real_image_dataset_test/'

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [4]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = AutoModel.from_pretrained("google/vit-base-patch16-224").to(DEVICE)

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
def infer(image):
    """
    Process an image using a pre-defined model and processor.
    
    Args:
        image (PIL.Image.Image): An image loaded in PIL format.
    
    Returns:
        torch.Tensor: The processed tensor output from the model's pooler layer.
    """
    inputs = processor(image, return_tensors="pt").to(DEVICE)
    outputs = model(**inputs)
    return outputs.pooler_output

def load_image(image_path):
    """
    Load an image from the specified file path and convert it to RGB.
    
    Args:
        image_path (str): Path to the image file.
    
    Returns:
        PIL.Image.Image: The loaded and converted image.
    """
    image = Image.open(image_path).convert('RGB')
    image = image.resize((256, 256), Image.LANCZOS)
    return image

def process_and_save_image(image_path, output_directory):
    """
    Load an image, process it with the model, and save the output to a JSON file.
    
    Args:
        image_path (str): The file path of the image to process.
        output_directory (str): Directory where the output JSON should be saved.
    """
    filename = os.path.basename(image_path)
    try:
        image = load_image(image_path)
        output = infer(image)
        # Convert the tensor output to a list for JSON serialization
        result = {
            "filename": filename,
            "embedding": output.tolist()
        }
        output_file_path = os.path.join(output_directory, f"{filename}.json")
        with open(output_file_path, 'w') as f:
            json.dump(result, f, indent=4)
    except Exception as e:
        print(f"Failed to process {filename}: {str(e)}")

def process_directory(directory_path):
    """
    Process all images in a directory and save their model outputs in a subdirectory.
    
    Args:
        directory_path (str): Path to the directory containing images.
    """
    # Create a subdirectory 'results' within the given directory to store outputs
    output_directory = os.path.join(directory_path, "results")
    os.makedirs(output_directory, exist_ok=True)
    # Process each file in the directory using a progress bar
    for filename in tqdm(os.listdir(directory_path)):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff')):
            image_path = os.path.join(directory_path, filename)
            process_and_save_image(image_path, output_directory)

In [6]:
def load_embeddings_from_jsons(directory_path):
    """
    Load embeddings from JSON files stored in a directory, displaying a progress bar, and create a DataFrame.
    
    Args:
        directory_path (str): The path to the directory containing JSON files.
    
    Returns:
        pd.DataFrame: A DataFrame where each row is an embedding vector and the index is the filename.
    """
    data = {}
    
    # Retrieve all JSON files in the directory
    json_files = [f for f in os.listdir(directory_path) if f.endswith('.json')]
    
    # Iterate through all files with a progress bar
    for filename in tqdm(json_files, desc="Loading JSON files"):
        file_path = os.path.join(directory_path, filename)
        with open(file_path, 'r') as file:
            json_data = json.load(file)
            # Assume 'filename' and 'embedding' are keys in the JSON
            data[json_data['filename']] = json_data['embedding']
    
    # Create DataFrame
    df = pd.DataFrame.from_dict(data, orient='index', columns=['Embedding'])
    return df

In [7]:
dataset_directory_path = os.getcwd() + '/test_data/eval_large_real_image_dataset_test/'
process_directory(dataset_directory_path)

100%|█████████████████████████████████████████████████████████████████████████████████| 993/993 [01:43<00:00,  9.57it/s]


In [8]:
jsons_directory_path = dataset_directory_path + '/results/'
embeddings_df = load_embeddings_from_jsons(jsons_directory_path)
embeddings_df

Loading JSON files: 100%|███████████████████████████████████████████████████████████| 990/990 [00:00<00:00, 3244.70it/s]


,Embedding
8363.jpg,"[0.5206978917121887, 0.0018596628215163946, -0..."
33514.jpg,"[0.4463893473148346, 0.4649040699005127, -0.26..."
22.jpg,"[-0.49667906761169434, 0.8522477746009827, -0...."
35.jpg,"[0.49608179926872253, 0.2353062480688095, -0.2..."
41864.jpg,"[0.6275619268417358, 0.17147286236286163, -0.6..."
...,...
41854.jpg,"[0.5204341411590576, -0.03668942674994469, 0.7..."
25138.jpg,"[0.005344682838767767, 0.14931190013885498, 0...."
75266.jpg,"[0.22541072964668274, 0.5299910306930542, 0.01..."
33490.jpg,"[-0.10613834857940674, -0.6904759407043457, 0...."


In [10]:
def create_cosine_similarity_matrix(df, normalize=True):
    """
    Create a cosine similarity matrix from the embeddings stored in the DataFrame.

    Args:
        df (pd.DataFrame): DataFrame containing embeddings in each row under the column 'Embedding' and file names under 'file_name'.
        normalize (bool): Whether to normalize embeddings before calculating cosine similarity.
    
    Returns:
        pd.DataFrame: A DataFrame representing the cosine similarity scores between embeddings, indexed by file names.
    """
    # Convert DataFrame column to a list of embeddings
    embeddings = df['Embedding'].tolist()
    
    # Convert list to a PyTorch tensor with dtype as float (important for decimal operations)
    embeddings_tensor = torch.tensor(embeddings, dtype=torch.float32)
    print("Embeddings Tensor Shape:", embeddings_tensor.shape)
    
    # Normalize embeddings if required
    if normalize:
        embeddings_tensor = torch.nn.functional.normalize(embeddings_tensor, p=2, dim=1)
    
    # Calculate cosine similarity matrix
    similarity_matrix = torch.mm(embeddings_tensor, embeddings_tensor.t())  # t() transposes the second tensor

    # Convert tensor to numpy array and then to DataFrame, using file names as row and column labels
    similarity_df = pd.DataFrame(similarity_matrix.numpy(), index=df.index, columns=df.index)
    
    return similarity_df


In [11]:
similarity_matrix = create_cosine_similarity_matrix(embeddings_df)
similarity_matrix

Embeddings Tensor Shape: torch.Size([990, 768])


,8363.jpg,33514.jpg,22.jpg,35.jpg,41864.jpg,83687.jpg,100399.jpg,58590.jpg,92014.jpg,33479.jpg,...,117139.jpg,91997.jpg,100350.jpg,8409.jpg,108771.jpg,41854.jpg,25138.jpg,75266.jpg,33490.jpg,50239.jpg
8363.jpg,1.000000,-0.001758,-0.087108,0.035488,-0.011625,0.044276,-0.010875,0.050954,0.016940,0.018511,...,0.041734,0.141689,-0.018504,0.209465,-0.038167,-0.040754,0.198403,0.024042,0.035711,0.028383
33514.jpg,-0.001758,1.000000,0.095446,0.092417,0.008267,0.070692,0.057626,-0.016555,0.097763,-0.006212,...,0.306826,0.179815,0.124180,-0.085339,0.086080,-0.007335,0.035200,0.143568,0.092363,0.039065
22.jpg,-0.087108,0.095446,1.000000,0.075565,0.234091,0.090871,0.033452,-0.048880,0.076640,-0.058437,...,-0.030467,0.008327,0.204452,0.008251,0.116218,0.086242,-0.035898,-0.000696,0.096482,0.017413
35.jpg,0.035488,0.092417,0.075565,1.000000,0.160309,0.007181,-0.011813,-0.062791,-0.042583,0.053971,...,0.212411,0.108910,0.071604,0.024369,0.048784,0.041325,-0.000628,0.089611,0.001578,0.056254
41864.jpg,-0.011625,0.008267,0.234091,0.160309,1.000000,0.083988,0.011385,0.023476,0.099720,0.017214,...,0.088222,0.077832,0.150669,-0.003960,0.017754,0.022338,-0.015410,0.087526,0.047827,0.087966
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41854.jpg,-0.040754,-0.007335,0.086242,0.041325,0.022338,0.022566,0.040707,0.101935,0.000660,-0.008423,...,0.023890,-0.013311,0.040598,-0.025840,0.100246,1.000000,0.033592,-0.028473,0.090097,0.080287
25138.jpg,0.198403,0.035200,-0.035898,-0.000628,-0.015410,0.039450,0.034599,-0.027860,-0.018924,-0.007854,...,0.111391,0.045035,0.087817,0.037938,-0.055378,0.033592,1.000000,0.046589,0.029474,-0.002134
75266.jpg,0.024042,0.143568,-0.000696,0.089611,0.087526,-0.063745,-0.041792,0.003475,-0.012303,-0.003142,...,0.258178,0.090257,0.125986,-0.016997,0.053523,-0.028473,0.046589,1.000000,0.036015,-0.075727
33490.jpg,0.035711,0.092363,0.096482,0.001578,0.047827,0.407736,-0.024317,0.041499,0.312853,0.125530,...,0.022679,-0.007199,0.043342,-0.041088,0.027974,0.090097,0.029474,0.036015,1.000000,0.037370


# LPIPs Image Pair Similarity Metric

In [26]:
import inspect
import os

In [55]:
# Test model loading
model_path = '/root/miniconda3/envs/bitmind/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth'
model = lpips.LPIPS(net='alex').cuda()
state_dict = torch.load(model_path, map_location='cpu')
model.load_state_dict(state_dict, strict=False)
model

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /root/miniconda3/envs/bitmind/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth


### Generated synthetic image directory

In [43]:
def load_image(path, resize_shape=None):
    """
    Load and optionally resize an image from the specified path using PIL.

    Args:
        path (str): The path to the image file.
        resize_shape (tuple of int, optional): The target size as (width, height) if resizing is desired.

    Returns:
        ndarray: The loaded (and possibly resized) image in RGB format.
    """
    # Load image with PIL
    img = PIL.Image.open(path)

    # Resize the image if a resize shape is provided
    if resize_shape is not None:
        img = img.resize(resize_shape, PIL.Image.LANCZOS)

    # Convert the image to RGB if not already in that mode
    if img.mode != 'RGB':
        img = img.convert('RGB')

    # Convert the image to an array and ensure it's 8-bit per channel
    img = np.array(img).astype('uint8')

    return img
    
def im2tensor(image, imtype=np.uint8, cent=1., factor=255./2.):
    return torch.Tensor((image / factor - cent)
                        [:, :, :, np.newaxis].transpose((3, 2, 0, 1)))

def isImageFile(filepath):
    return filepath.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff', '.bmp', '.gif'))

In [19]:
def compute_image_directory_lpip(directory, output_file, max_images=None, use_gpu=False, all_pairs=False, resize_shape=(256, 256)):
    """
    Calculate and write the perceptual distance metric between images in a specified directory using LPIPS.

    Args:
        directory (str): Path to the directory containing images to compare.
        output_file (str): File path where the LPIP distance will be written.
        max_images (int, optional): Maximum number of images to process. If None, processes all images.
        use_gpu (bool, optional): If True, utilizes GPU to accelerate the computation.
        all_pairs (bool, optional): If True, computes distances for all pairs of images. If False, computes only for sequential pairs.

    Returns:
        None: This function writes the results directly to the output file and does not return any value.
    """
    # Initialize the LPIPS model with a predefined neural network (e.g., AlexNet).
    if use_gpu:
        loss_fn = lpips.LPIPS(net='alex').cuda()
    else: loss_fn = lpips.LPIPS(net='alex')

    # Open the output file for writing results
    with open(directory+output_file, 'w') as f:
        files = sorted(os.listdir(directory))
        if max_images is not None:
            files = files[:max_images]

        dists = []  # List to hold computed distances

        total_comparisons = sum(len(files[i+1:]) if all_pairs else 1 for i in range(len(files)-1))
        completed_comparisons = 0
        start_time = time.time()

        # Process each file in the directory
        for i, file in enumerate(files[:-1]):
            if not isImageFile(os.path.join(directory, file)):
                continue
            img0 = lpips.im2tensor(load_image(os.path.join(directory, file), resize_shape))
            if use_gpu:
                img0 = img0.cuda()  # Transfer tensor to GPU if enabled

            # Determine comparison targets based on 'all_pairs' flag
            if all_pairs:
                files1 = files[i+1:]
            else:
                files1 = [files[i+1]] if i + 1 < len(files) else []

            for file1 in files1:
                if not isImageFile(os.path.join(directory, file1)):
                    continue
                img1 = lpips.im2tensor(load_image(os.path.join(directory, file1), resize_shape))
                if use_gpu:
                    img1 = img1.cuda()

                # Compute the perceptual metric between two images
                dist01 = loss_fn.forward(img0, img1)
                f.write('({}, {}): {:.6f}\n'.format(file, file1, dist01.item()))

                dists.append(dist01.item())

                # Update progress
                completed_comparisons += 1
                percentage_complete = (completed_comparisons / total_comparisons) * 100
                elapsed_time = time.time() - start_time
                estimated_total_time = elapsed_time / completed_comparisons * total_comparisons
                estimated_time_remaining = estimated_total_time - elapsed_time
                eta = datetime.datetime.now() + datetime.timedelta(seconds=estimated_time_remaining)
            
                # Convert seconds to hours, minutes, and seconds
                hours, remainder = divmod(estimated_time_remaining, 3600)
                minutes, seconds = divmod(remainder, 60)

                if completed_comparisons % (total_comparisons // 100) == 0 or completed_comparisons == total_comparisons:
                    print(f"Progress: {percentage_complete:.2f}% - Time Remaining: {int(hours):02}:{int(minutes):02}:{int(seconds):02}")
                    print(f"ETA: {eta.strftime('%Y-%m-%d %H:%M:%S')}")

        # Calculate statistics: average distance and standard error
        try:
            avg_dist = np.mean(dists)
            stderr_dist = np.std(dists) / np.sqrt(len(dists))
            print('Avg: {:.5f} +/- {:.5f}'.format(avg_dist, stderr_dist))
            f.write('Avg: {:.6f} +/- {:.6f}'.format(avg_dist, stderr_dist))
        except Exception as e:
            print('Error:', e)

In [33]:
directory_path

'/root/bitmind-subnet/bitmind/synthetic_image_generation/data/eval_large_real_image_dataset_test/'

In [20]:
%%time
compute_image_directory_lpip(directory_path, 'lpip_metrics', max_images=200, use_gpu=True, all_pairs=True, resize_shape=(256,256))

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /root/miniconda3/envs/bitmind/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth
Progress: 1.00% - ETA: 2024-07-10 20:35:14
Progress: 2.00% - ETA: 2024-07-10 20:34:25
Progress: 3.00% - ETA: 2024-07-10 20:34:13
Progress: 4.00% - ETA: 2024-07-10 20:34:14
Progress: 5.00% - ETA: 2024-07-10 20:34:07
Progress: 6.00% - ETA: 2024-07-10 20:34:04
Progress: 7.00% - ETA: 2024-07-10 20:34:00
Progress: 8.00% - ETA: 2024-07-10 20:33:55
Progress: 9.00% - ETA: 2024-07-10 20:33:59
Progress: 10.00% - ETA: 2024-07-10 20:33:58
Progress: 11.00% - ETA: 2024-07-10 20:33:50
Progress: 12.00% - ETA: 2024-07-10 20:33:46
Progress: 13.00% - ETA: 2024-07-10 20:33:47
Progress: 14.00% - ETA: 2024-07-10 20:33:43
Progress: 15.00% - ETA: 2024-07-10 20:33:43
Progress: 16.00% - ETA: 2024-07-10 20:33:42
Progress: 17.00% - ETA: 2024-07-10 20:33:35
Progress: 18.00% - ETA: 2024-07-10 20:33:33
Progress: 19.00% - ETA: 2024-07-10 20: